In [ ]:
import pandas as pd
import os
from imblearn.under_sampling import RandomUnderSampler

# === 1. Load Split Datasets ===
split_dir = "../data/splits"

train_path = os.path.join(split_dir, "train.csv")

train_df = pd.read_csv(train_path)

print(" Loaded split datasets:")
print(f" - train.csv: {len(train_df):,}")


 Loaded split datasets:
 - train.csv: 4,453,834


In [6]:
# === 2. Check class imbalance before downsampling ===
n_pos = train_df["isFraud"].sum()
n_neg = (train_df["isFraud"] == 0).sum()
print(f"\nBefore downsampling:")
print(f"  Fraud cases:     {n_pos:,}")
print(f"  Non-fraud cases: {n_neg:,}")
print(f"  Fraud rate:      {train_df['isFraud'].mean():.5f}")


Before downsampling:
  Fraud cases:     5,749
  Non-fraud cases: 4,448,085
  Fraud rate:      0.00129


In [7]:
# === 3. Apply RandomUnderSampler (1:10 ratio) ===
RATIO = 10
rng = 42 #set seed

X_train = train_df.drop(columns=["isFraud"])
y_train = train_df["isFraud"]

target_neg = min(RATIO * n_pos, n_neg)
sampling_strategy = {0: target_neg, 1: n_pos}

rus = RandomUnderSampler(sampling_strategy=sampling_strategy,
                         random_state=rng, replacement=False)

X_down, y_down = rus.fit_resample(X_train, y_train)

train_downsampled = pd.concat([X_down, y_down], axis=1).sample(frac=1, random_state=rng).reset_index(drop=True)


In [8]:

# === 4. Check class balance after downsampling ===
n_pos_after = train_downsampled["isFraud"].sum()
n_neg_after = (train_downsampled["isFraud"] == 0).sum()

print("\nAfter downsampling (1:10):")
print(f"  Fraud cases:     {n_pos_after:,}")
print(f"  Non-fraud cases: {n_neg_after:,}")
print(f"  Fraud rate:      {train_downsampled['isFraud'].mean():.4f} (expected ≈ {1/(1+RATIO):.4f})")
print(f"  Total samples:   {len(train_downsampled):,}")

# === 5. Save Downsampled Training Set ===
out_path = os.path.join(split_dir, "train_downsampled_1to10.csv")
train_downsampled.to_csv(out_path, index=False)

print(f"\n Saved downsampled training set to: {out_path}")



After downsampling (1:10):
  Fraud cases:     5,749
  Non-fraud cases: 57,490
  Fraud rate:      0.0909 (expected ≈ 0.0909)
  Total samples:   63,239

 Saved downsampled training set to: ../data/splits/train_downsampled_1to10.csv


In [11]:
print("\n=== CHECKING REPRESENTATIVENESS AFTER DOWNSAMPLING ===")

# Compare fraud/non-fraud counts before vs after
print("\nFraud vs Non-Fraud Counts:")
print(pd.DataFrame({
    'Before': train_df['isFraud'].value_counts(),
    'After': train_downsampled['isFraud'].value_counts()
}).rename(index={0: 'Non-Fraud', 1: 'Fraud'}))

# Compare transaction type proportions
print("\nTransaction Type Distribution (Before vs After):")
type_dist_before = train_df['type'].value_counts(normalize=True) * 100
type_dist_after = train_downsampled['type'].value_counts(normalize=True) * 100
type_comparison = pd.concat([type_dist_before, type_dist_after], axis=1)
type_comparison.columns = ['Before (%)', 'After (%)']
print(type_comparison.round(2))

# Compare descriptive statistics of numeric features
numeric_cols = train_df.select_dtypes(include=['float64', 'int64']).columns.drop('isFraud')
desc_before = train_df[numeric_cols].describe().loc[['mean', 'std']]
desc_after = train_downsampled[numeric_cols].describe().loc[['mean', 'std']]
desc_diff = (desc_after - desc_before).round(3)

print("\nNumeric Feature Means/STD Change (After - Before):")
print(desc_diff.T)



=== CHECKING REPRESENTATIVENESS AFTER DOWNSAMPLING ===

Fraud vs Non-Fraud Counts:
            Before  After
isFraud                  
Non-Fraud  4448085  57490
Fraud         5749   5749

Transaction Type Distribution (Before vs After):
          Before (%)  After (%)
type                           
CASH_OUT       35.16      36.43
PAYMENT        33.82      30.98
CASH_IN        22.00      19.87
TRANSFER        8.38      12.13
DEBIT           0.65       0.59

Numeric Feature Means/STD Change (After - Before):
                      mean         std
step                11.224      11.892
amount          120474.649  432742.563
oldbalanceOrg    54531.449   36705.621
newbalanceOrig  -79688.766 -111177.523
oldbalanceDest  -19700.071  457715.732
newbalanceDest   37619.314  493125.182
isFlaggedFraud       0.000       0.013
